___
TEXT MINING - SENTENCE EMBEDDINGS
___

# Modern Text Embeddings with Sentence-Transformers

In this notebook we'll explore **modern embeddings** using the [Sentence-Transformers](https://www.sbert.net/) library, which is completely **open source and free**.

Unlike Word2Vec and GloVe which generate word-level embeddings, transformer models generate **sentence/document-level embeddings**, better capturing semantic context.

## Why Sentence-Transformers?

- **Free**: No API key or costs required
- **Open Source**: Models available on Hugging Face
- **State-of-the-art**: Based on Transformer architecture (like BERT, RoBERTa)
- **Easy to use**: Simple and intuitive API
- **Multilingual**: Supports many languages including Italian

## Installation

Let's install the required libraries:

In [ ]:
!pip install sentence-transformers -q

: 

## 1. Loading the Model

We'll use `all-MiniLM-L6-v2`, a lightweight but effective model:
- **Embedding dimension**: 384
- **Speed**: Very fast
- **Quality**: Excellent for most use cases

In [ ]:
from sentence_transformers import SentenceTransformer

# Load a lightweight and fast model
model = SentenceTransformer('all-MiniLM-L6-v2')

print(f"Model loaded!")
print(f"Embedding dimension: {model.get_sentence_embedding_dimension()}")

## 2. Generating Embeddings

Let's see how to generate embeddings for individual sentences:

In [ ]:
# Generate the embedding for a sentence
sentence = "The quick brown fox jumps over the lazy dog."
embedding = model.encode(sentence)

print(f"Sentence: '{sentence}'")
print(f"\nEmbedding shape: {embedding.shape}")
print(f"\nFirst 10 values: {embedding[:10]}")

In [ ]:
# We can generate embeddings for multiple sentences at once
sentences = [
    "I love machine learning",
    "Deep learning is fascinating",
    "I enjoy eating pizza",
    "The weather is nice today"
]

embeddings = model.encode(sentences)

print(f"Number of sentences: {len(sentences)}")
print(f"Embeddings shape: {embeddings.shape}")

## 3. Computing Semantic Similarity

Embeddings allow us to calculate how semantically similar two sentences are using **cosine similarity**:

In [ ]:
from sentence_transformers import util

# Compute the similarity matrix
similarity_matrix = util.cos_sim(embeddings, embeddings)

print("Similarity Matrix:\n")
print(similarity_matrix.numpy().round(3))

In [ ]:
# Visualize the matrix as a heatmap
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

plt.figure(figsize=(10, 8))
sns.heatmap(
    similarity_matrix.numpy(), 
    annot=True, 
    fmt='.2f',
    xticklabels=[f"S{i+1}" for i in range(len(sentences))],
    yticklabels=[s[:30] + "..." if len(s) > 30 else s for s in sentences],
    cmap='RdYlGn'
)
plt.title('Semantic Similarity Matrix')
plt.tight_layout()
plt.show()

As we can see:
- "I love machine learning" and "Deep learning is fascinating" have high similarity (both talk about ML/AI)
- "I enjoy eating pizza" and "The weather is nice today" are less correlated to the technical sentences

## 4. Semantic Search

A practical application: finding the most similar documents to a query.

In [ ]:
# Document corpus
corpus = [
    "Python is a popular programming language for data science.",
    "Machine learning models can predict future outcomes.",
    "The Eiffel Tower is located in Paris, France.",
    "Neural networks are inspired by the human brain.",
    "Pizza originated in Naples, Italy.",
    "Natural language processing helps computers understand text.",
    "The Great Wall of China is a famous landmark.",
    "Deep learning has revolutionized computer vision."
]

# Generate embeddings for the entire corpus
corpus_embeddings = model.encode(corpus, convert_to_tensor=True)

print(f"Corpus encoded: {corpus_embeddings.shape}")

In [ ]:
# Search query
query = "How do artificial intelligence systems work?"
query_embedding = model.encode(query, convert_to_tensor=True)

# Find the most similar documents
hits = util.semantic_search(query_embedding, corpus_embeddings, top_k=5)[0]

print(f"Query: '{query}'\n")
print("Most relevant results:\n")
for i, hit in enumerate(hits, 1):
    print(f"{i}. [Score: {hit['score']:.3f}] {corpus[hit['corpus_id']]}")

## 5. Visualizing Embeddings with t-SNE

Let's reduce dimensionality to visualize embeddings in 2D:

In [ ]:
from sklearn.manifold import TSNE
import pandas as pd

# Larger dataset for visualization
texts = [
    # Tech/AI
    "Machine learning is transforming industries",
    "Deep learning uses neural networks",
    "Artificial intelligence can solve complex problems",
    "Python is great for data science",
    "Algorithms process large datasets",
    # Food
    "I love eating Italian pasta",
    "Pizza is my favorite food",
    "Cooking healthy meals is important",
    "Fresh vegetables are nutritious",
    "Chocolate cake is delicious",
    # Travel
    "Paris is a beautiful city",
    "I want to visit Japan",
    "Beach vacations are relaxing",
    "Mountain hiking is adventurous",
    "Exploring new cultures is exciting"
]

categories = ['Tech']*5 + ['Food']*5 + ['Travel']*5

# Generate embeddings
text_embeddings = model.encode(texts)

print(f"Embeddings generated: {text_embeddings.shape}")

In [ ]:
# Reduce to 2D with t-SNE
tsne = TSNE(n_components=2, random_state=42, perplexity=5)
embeddings_2d = tsne.fit_transform(text_embeddings)

# Create a DataFrame for visualization
df = pd.DataFrame({
    'x': embeddings_2d[:, 0],
    'y': embeddings_2d[:, 1],
    'category': categories,
    'text': texts
})

# Visualize
plt.figure(figsize=(12, 8))
colors = {'Tech': 'blue', 'Food': 'green', 'Travel': 'red'}

for category in df['category'].unique():
    mask = df['category'] == category
    plt.scatter(df[mask]['x'], df[mask]['y'], 
                c=colors[category], label=category, s=100, alpha=0.7)

# Add labels
for idx, row in df.iterrows():
    plt.annotate(row['text'][:25] + '...', 
                 (row['x'], row['y']),
                 fontsize=8, alpha=0.8)

plt.legend()
plt.title('t-SNE Visualization of Embeddings')
plt.xlabel('Dimension 1')
plt.ylabel('Dimension 2')
plt.tight_layout()
plt.show()

## 6. Multilingual Models

Sentence-Transformers also offers multilingual models that support Italian and many other languages:

In [ ]:
# Load a multilingual model
multilingual_model = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')

# Texts in different languages with the same meaning
texts_multilingual = [
    "I love artificial intelligence",      # English
    "Amo l'intelligenza artificiale",      # Italian
    "J'adore l'intelligence artificielle", # French
    "Me encanta la inteligencia artificial", # Spanish
    "Ich liebe künstliche Intelligenz",    # German
    "The weather is beautiful today",      # Different sentence (English)
    "Oggi il tempo è bellissimo"           # Different sentence (Italian)
]

# Generate embeddings
multilingual_embeddings = multilingual_model.encode(texts_multilingual)

# Compute similarity
multilingual_similarity = util.cos_sim(multilingual_embeddings, multilingual_embeddings)

print("Similarity between multilingual sentences:\n")
for i, text_i in enumerate(texts_multilingual):
    for j, text_j in enumerate(texts_multilingual):
        if i < j:
            sim = multilingual_similarity[i][j].item()
            print(f"[{sim:.3f}] '{text_i[:40]}...' <-> '{text_j[:40]}...'")

In [ ]:
# Visualize the similarity matrix
plt.figure(figsize=(12, 10))
labels = [t[:20] + "..." for t in texts_multilingual]

sns.heatmap(
    multilingual_similarity.numpy(),
    annot=True,
    fmt='.2f',
    xticklabels=labels,
    yticklabels=labels,
    cmap='RdYlGn'
)
plt.title('Cross-Lingual Similarity')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

**Observation**: Sentences talking about "artificial intelligence" in different languages have high similarity with each other, demonstrating that the model captures semantic meaning regardless of language!

## 7. Comparing Different Models

Let's try different models available for free:

In [ ]:
# List of models to compare (all free!)
model_names = [
    'all-MiniLM-L6-v2',           # Fast and lightweight (384 dim)
    'all-mpnet-base-v2',          # More accurate (768 dim)
    'paraphrase-MiniLM-L6-v2',    # Great for paraphrasing
]

test_sentences = [
    "The cat sat on the mat",
    "A feline rested on the rug",  # Paraphrase of the first
    "Dogs are loyal animals"        # Different sentence
]

print("Model comparison:\n")
print(f"Sentence 1: {test_sentences[0]}")
print(f"Sentence 2: {test_sentences[1]} (paraphrase)")
print(f"Sentence 3: {test_sentences[2]} (different)\n")

for model_name in model_names:
    test_model = SentenceTransformer(model_name)
    test_embeddings = test_model.encode(test_sentences)
    
    sim_1_2 = util.cos_sim(test_embeddings[0], test_embeddings[1]).item()
    sim_1_3 = util.cos_sim(test_embeddings[0], test_embeddings[2]).item()
    
    print(f"Model: {model_name}")
    print(f"  Similarity (1-2 paraphrase): {sim_1_2:.3f}")
    print(f"  Similarity (1-3 different):  {sim_1_3:.3f}")
    print(f"  Dimension: {test_model.get_sentence_embedding_dimension()}\n")

## 8. Clustering with Embeddings

Let's use embeddings to automatically group similar texts:

In [ ]:
from sklearn.cluster import KMeans

# Dataset for clustering
documents = [
    # Sports
    "The football match ended in a draw",
    "Tennis players compete at Wimbledon",
    "Basketball requires teamwork and skill",
    # Technology
    "Smartphones have changed communication",
    "Artificial intelligence is advancing rapidly",
    "Cloud computing enables remote work",
    # Nature
    "Forests are essential for biodiversity",
    "Ocean pollution threatens marine life",
    "Climate change affects weather patterns"
]

# Generate embeddings
doc_embeddings = model.encode(documents)

# Apply K-Means clustering
n_clusters = 3
kmeans = KMeans(n_clusters=n_clusters, random_state=42, n_init=10)
clusters = kmeans.fit_predict(doc_embeddings)

# Display results
print("Clustering Results:\n")
for cluster_id in range(n_clusters):
    print(f"\nCluster {cluster_id}:")
    for doc, cluster in zip(documents, clusters):
        if cluster == cluster_id:
            print(f"  - {doc}")

## 9. Practical Use Case: Duplicate Detection

Let's find duplicate or near-duplicate sentences in a dataset:

In [ ]:
# Dataset with potential duplicates
corpus_with_duplicates = [
    "How do I reset my password?",
    "I forgot my password, how can I reset it?",
    "What are your business hours?",
    "When is your store open?",
    "I need help with my order",
    "Can you assist me with a purchase issue?",
    "The product arrived damaged",
    "My item was broken when delivered"
]

# Generate embeddings
dup_embeddings = model.encode(corpus_with_duplicates, convert_to_tensor=True)

# Find similar pairs (threshold > 0.7)
threshold = 0.7
similarity_matrix = util.cos_sim(dup_embeddings, dup_embeddings)

print(f"Potential duplicates (similarity > {threshold}):\n")
for i in range(len(corpus_with_duplicates)):
    for j in range(i+1, len(corpus_with_duplicates)):
        sim = similarity_matrix[i][j].item()
        if sim > threshold:
            print(f"[{sim:.3f}]")
            print(f"  1: {corpus_with_duplicates[i]}")
            print(f"  2: {corpus_with_duplicates[j]}\n")

## Summary

In this notebook we covered:

1. **Sentence-Transformers**: Open source library for sentence embeddings
2. **Generating embeddings**: How to convert text to numerical vectors
3. **Semantic similarity**: Computing cosine similarity between embeddings
4. **Semantic search**: Finding relevant documents
5. **Visualization**: t-SNE to visualize embeddings in 2D
6. **Multilingual models**: Support for Italian and other languages
7. **Clustering**: Automatic grouping of texts
8. **Duplicate detection**: Finding similar/duplicate texts

### Recommended Models (all free!):

| Model | Dimension | Speed | Use Case |
|-------|-----------|-------|----------|
| `all-MiniLM-L6-v2` | 384 | Fast | General purpose |
| `all-mpnet-base-v2` | 768 | Medium | Best quality |
| `paraphrase-multilingual-MiniLM-L12-v2` | 384 | Medium | Multilingual |

### Resources:
- [Sentence-Transformers Documentation](https://www.sbert.net/)
- [Hugging Face Model Hub](https://huggingface.co/models?library=sentence-transformers)
- [Pretrained Models](https://www.sbert.net/docs/pretrained_models.html)